In [2]:
import yfinance as yf
import pandas as pd
import numpy as np
from scipy.stats import norm
import plotly.express as px
import plotly.graph_objects as go

## Obtención y Preparación de Datos

In [3]:
# Descargar los datos
sp500 = yf.download("^GSPC", start="2018-01-01")
sp500.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,^GSPC,^GSPC,^GSPC,^GSPC,^GSPC
Date,,,,,
2018-01-02,2695.810059,2695.889893,2682.360107,2683.729980,3397430000
2018-01-03,2713.060059,2714.370117,2697.770020,2697.850098,3544030000
2018-01-04,2723.989990,2729.290039,2719.070068,2719.310059,3697340000
2018-01-05,2743.149902,2743.449951,2727.919922,2731.330078,3239280000
2018-01-08,2747.709961,2748.510010,2737.600098,2742.669922,3246160000


In [4]:
# Convertir la serie a DataFrame
df_close=sp500[("Close","^GSPC")].to_frame()
df_close.head()

,Close
,^GSPC
Date,
2018-01-02,2695.810059
2018-01-03,2713.060059
2018-01-04,2723.989990
2018-01-05,2743.149902
2018-01-08,2747.709961


In [5]:
print(type(df_close))
print(df_close.columns)

<class 'pandas.DataFrame'>
MultiIndex([('Close', '^GSPC')],
           )


In [6]:
# Quitar el MultiIndex
df_close.columns = ['Close']
df_close.head()

,Close
Date,
2018-01-02,2695.810059
2018-01-03,2713.060059
2018-01-04,2723.989990
2018-01-05,2743.149902
2018-01-08,2747.709961


In [7]:
# Verificar valores faltantes
df_close.isna().sum()

Close    0
dtype: int64

## Rendimientos

In [8]:
# Rendimientos
df_close['Simple_Return'] = (df_close['Close'] - df_close['Close'].shift())/df_close['Close'].shift()
df_close.head(5)

,Close,Simple_Return
Date,,
2018-01-02,2695.810059,NaN
2018-01-03,2713.060059,0.006399
2018-01-04,2723.989990,0.004029
2018-01-05,2743.149902,0.007034
2018-01-08,2747.709961,0.001662


In [9]:
# Log-rendimientos
df_close['Log_Return'] = np.log(df_close['Close']/df_close['Close'].shift())
df_close.head(5)

,Close,Simple_Return,Log_Return
Date,,,
2018-01-02,2695.810059,NaN,NaN
2018-01-03,2713.060059,0.006399,0.006378
2018-01-04,2723.989990,0.004029,0.004021
2018-01-05,2743.149902,0.007034,0.007009
2018-01-08,2747.709961,0.001662,0.001661


## Exploración Visual

In [10]:
# Gráfica del precio de cierre
close_graph = px.line(x = df_close.index, y = df_close['Close'])
close_graph.update_layout(title={'text': 'S&P 500 Closing Price', 'x':0.5})
close_graph.update_xaxes(title_text='Date')
close_graph.update_yaxes(title_text='Price')


In [11]:
# Gráfica el Rendimiento Simple
return_graph = px.line(x = df_close.index, y = df_close['Simple_Return'])
return_graph.update_layout(title={'text': 'S&P 500 Simple Return', 'x':0.5})
return_graph.update_xaxes(title_text='Date')
return_graph.update_yaxes(title_text='Return')

In [12]:
# Gráfica el Log-Rendimiento
logreturn_graph = px.line(x = df_close.index, y = df_close['Log_Return'])
logreturn_graph.update_layout(title={'text': 'S&P 500 Log-Return', 'x':0.5})
logreturn_graph.update_xaxes(title_text='Date')
logreturn_graph.update_yaxes(title_text='Return')

### Observaciones

A simple vista es fácil notar que las gráficas de rendimientos tienen una media más estable a través del tiempo, lo cuál hace más útiles estos datos para trabajar con modelos estadísticos.

## Estadística Descriptiva

In [13]:
print('Simple Return Mean:', round(df_close['Simple_Return'].mean(),5))
print('Simple Return Median:', round(df_close['Simple_Return'].median(),5))
print('Simple Return Standard Deviation:', round(df_close['Simple_Return'].std(),5))
print('Simple Return Minimum:', round(df_close['Simple_Return'].min(),5))
print('Simple Return Maximum:', round(df_close['Simple_Return'].max(),5))
print('Simple Return Skewness:', round(df_close['Simple_Return'].skew(),5))
print('Simple Return Kurtosis:', round(df_close['Simple_Return'].kurtosis(),5))

Simple Return Mean: 0.00055
Simple Return Median: 0.00091
Simple Return Standard Deviation: 0.01218
Simple Return Minimum: -0.11984
Simple Return Maximum: 0.09515
Simple Return Skewness: -0.35245
Simple Return Kurtosis: 14.0001


In [14]:
df_close['Simple_Return'].describe()

count    2132.000000
mean        0.000551
std         0.012177
min        -0.119841
25%        -0.004371
50%         0.000905
75%         0.006480
max         0.095154
Name: Simple_Return, dtype: float64

### Interpretación

- Se observan rendimientos diarios promedio del 0.05%.
- La mediana es mayor que la media, lo que podría indicar que la magnitud de los eventos extremos en la cola izquierda fue mayor.
- La desviación estándar es de 1.2% lo que indica que en promedio, el rendimiento diario se encontraba en un intervalo de $\pm$ 1.2 puntos porcentuales respecto a la media.
- El rendimiento mínimo observado fue de -11.98%.
- El rendimiento máximo observado fue de 9.51%.
- Tenemos una asimetría de -0.354 lo que indica que la distribución tiene una cola izquirda más pesada y por tanto, los eventos negativos extremos tienden a ser más significativos que los positivos.
- Tenemos una curtosis de 14, lo que nos indica que los eventos extremos de los rendimientos del índice no son los que se esperarían si estos siguieran una distribución normal.

## Distribución de Rendimientos

In [15]:
return_hist = px.histogram(df_close['Simple_Return'], histnorm= 'probability density',nbins=100)

return_hist.update_traces(
    marker_color="#3f1fb4",
    marker_line_color='black',
    marker_line_width=0.7
)

return_hist.update_layout(
    title_text="S&P 500 Simple Return Histogram",
    title_x=0.5,
    title_font=dict(size=18, color='#333333'),
    template='simple_white',
    showlegend=False,
    width=800,
    height=450
)

return_hist.update_xaxes(title_text="Return Rate", title_font=dict(size=15, color='#333333'),  
                         showgrid=True, gridcolor='#f0f0f0', zeroline=True, zerolinecolor='black')
return_hist.update_yaxes(title_text="Probability Density", title_font=dict(size=15, color='#333333'), 
                         showgrid=True, gridcolor='#f0f0f0')


In [16]:
mu = df_close['Simple_Return'].mean()
sigma = df_close['Simple_Return'].std()
min_sr = df_close['Simple_Return'].min()
max_sr = df_close['Simple_Return'].max()

In [17]:
x_sr = np.linspace(min_sr, max_sr, 2100)
y_sr = norm.pdf(x_sr, mu, sigma)

In [18]:
return_hist.add_trace(
    go.Scatter(
        x=x_sr, 
        y=y_sr, 
        mode='lines',
        name="Normal Distribution",
        line=dict(color='red', width=1.5)
    )
)

return_hist.update_layout(
    template="simple_white",
    showlegend=True,  
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01), 
    title_x=0.5,
)

### Interpretación

Podemos observar que la distribución de los rendimientos no encaja con una distribución normal ya que aunque la mayoría de los datos se concetran en el centro, la densidad de estos es mayor a la que se esperaría de una normal, además podemos ver que tenemos eventos extremos muy severos y que junto con el valor de asimetría obtenido la cola izquierda es más pesada. Estos dos fenómenos tampoco se esperarían en una distribución normal

## Volatilidad 

Para esta sección se utilizará

$$\sigma_{anual} = \sqrt{252}\sigma_{diaria}$$

In [21]:
annual_simplereturn_std = np.sqrt(252) * df_close['Simple_Return'].std()
print('Volatilidad:', annual_simplereturn_std)

Volatilidad: 0.19330188547936708


In [28]:
annual_simplereturn_rw_std = np.sqrt(252) * df_close['Simple_Return'].rolling(window=30).std()
annual_simplereturn_rw_std.head()

Date
2018-01-02   NaN
2018-01-03   NaN
2018-01-04   NaN
2018-01-05   NaN
2018-01-08   NaN
Name: Simple_Return, dtype: float64

In [29]:
annual_simplereturn_rw_std.tail()

Date
2026-06-23    0.152735
2026-06-24    0.152637
2026-06-25    0.152579
2026-06-26    0.151515
2026-06-29    0.153845
Name: Simple_Return, dtype: float64

In [31]:
# Gráfica de la volatilidad móvil de los rendimientos simples
annual_simplereturn_rw_std_graph = px.line(x = annual_simplereturn_rw_std.index, y = annual_simplereturn_rw_std)
annual_simplereturn_rw_std_graph.update_layout(title={'text': 'S&P 500 Annualized Volatility (30-Day Rolling)', 'x':0.5})
annual_simplereturn_rw_std_graph.update_xaxes(title_text='Date')
annual_simplereturn_rw_std_graph.update_yaxes(title_text='Volatility')